In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [4]:
df=pd.read_csv('/content/drive/MyDrive/covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [7]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [8]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [10]:
X_train

,age,gender,fever,cough,city
65,69,Female,102.0,Mild,Bangalore
43,22,Female,99.0,Mild,Bangalore
67,65,Male,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
6,14,Male,101.0,Strong,Bangalore
...,...,...,...,...,...
16,69,Female,103.0,Mild,Kolkata
60,24,Female,102.0,Strong,Bangalore
10,75,Female,NaN,Mild,Delhi
40,49,Female,102.0,Mild,Delhi


### **Without Column Transformation**

In [11]:
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])

In [12]:
X_test_fever=si.fit_transform(X_test[['fever']])

In [13]:
X_train_fever.shape

(80, 1)

In [14]:
oe=OrdinalEncoder(categories=[['Mild','Strong']])

X_train_cough=oe.fit_transform(X_train[['cough']])
X_test_cough=oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

One Hot Encoding over Gender & City

In [15]:
ohe=OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city=ohe.fit_transform(X_test[['gender','city']])
X_train_gender_city.shape

X_train_gender_city.shape

(80, 4)

In [16]:
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age=X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape

(80, 1)

In [20]:
X_train_transformed=np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

In [21]:
X_test_transformed=np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

In [22]:
X_train_transformed.shape

(80, 7)

### **Column Transformer**

In [23]:
from sklearn.compose import ColumnTransformer

In [30]:
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city']),
],remainder='passthrough')

In [31]:
transformer.fit_transform(X_train)

array([[102.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  69.        ],
       [ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  22.        ],
       [ 99.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  65.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  56.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  14.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  69.        ],
       [103.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  16.        ],
       [ 98.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  83.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.    

In [32]:
transformer.fit_transform(X_test)

array([[102.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  64.        ],
       [100.31578947,   0.        ,   1.        ,   0.        ,
          0.        ,   1.        ,  23.        ],
       [104.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   6.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  64.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  13.        ],
       [101.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  34.        ],
       [ 99.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  59.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  81.        ],
       [100.        ,   0.        ,   1.        ,   1.        ,
          0.    

In [33]:
transformer.fit_transform(X_test).shape

(20, 7)